## Self-Attention




The previous section argued that a model for processing text will 

(i) use <mark>parameter sharing</mark> to cope with long input passages of differing lengths and 
(ii) contain <mark>connections between word representations</mark> that depend on the words themselves. 

The transformer acquires both properties by using <u>dot-product self-attention</u>.

<!-- A standard neural network layer $\mathbf{f[x]}$, takes a $D\times 1$ input $\mathbf{x}$ and applies a linear
transformation followed by an activation function like a ReLU, so:

\begin{eqnarray}
  \mbox{{\bf f}}[\mathbf{x}] = \mbox{{\bf ReLU}}[\boldsymbol\beta +\boldsymbol\Omega\mathbf{x}],
 \end{eqnarray}

where $\boldsymbol\beta$ contains the biases, and $\boldsymbol\Omega$ contains the weights. -->

### Value Vectors 

Value vectors in the context of self-attention are adjusted embeddings of the input tokens, capturing their information or semantics in a form that can be effectively combined with other tokens' information (through the attention mechanism). In other words, the value vector still satisfy the property that similar input tokens have similar value vectors. 

First, a set of <mark>Value vectors</mark> are computed for each input:


\begin{eqnarray}\label{eq:transformer_values}
  \mathbf{v}_{m} = \boldsymbol\beta_{v}+\boldsymbol\Omega_{v}\mathbf{x}_{m},
 \end{eqnarray}

where $\boldsymbol\beta_{v} ∈ \mathbb{R}^{D×1}$ and $\boldsymbol\Omega_{v} ∈ \mathbb{R}^{D×D}$ represent biases and weights, respectively.

Note that the subscript $m$ in $\mathbf{v}_{m}$ corresponds to the mth input $\mathbf{x}_{m}$ and $m \in \{1, \ldots, N\}$. More importantly note that only the value vector $\mathbf{v}_{m}$ and input vector $\mathbf{x}_{m}$ have the subscript $m$. Both beta and Omega do not have the subscript $m$ but have the subscripts $v$ to indicate that they are associated with the value vectors and not specific to any particular input. In other words, <mark>the same parameters $\boldsymbol\beta_{v}$ and $\boldsymbol\Omega_{v}$ are shared by all inputs.</mark>

<center><img src="valuevectors.jpg" style="filter: invert(1);" width="50%"></center>

Contrast this with a fully connected layer. In case of a fully connected layer, the weight matrix $\Omega$ would have been dense and of dimension $(N\times D) \times (N\times D)$. In self-attention, however, the weights $\boldsymbol\Omega_{v}$ are shared across all inputs, which allows the model to maintain permutation invariance and efficiently handle variable-length sequences.


Therefore, the equation $\mathbf{v}_{m} = \boldsymbol\beta_{v}+\boldsymbol\Omega_{v}\mathbf{x}_{m}$ shows that the same weights $\Omega_v ∈ \mathbb{R}^{D×D}$ and biases $\beta_v ∈ \mathbb{R}^{D}$ are applied to each input $x_• \in \mathbb{R}^{D}$. 

This computation scales linearly with the sequence length N, so it needs fewer parameters than a fully connected network relating all DN inputs to all DN values. 

In fact, the value computation can be viewed as a sparse matrix operation with shared parameters that relates these DN quantities (figure 12.2b).



### Self-Attention Block 

<mark>A self-attention block $\mathbf{sa}[•]$ takes N inputs $\mathbf{x}_1,...,\mathbf{x}_N$, each of dimension $D\times 1$,<br/>and returns N outputs $\mathbf{sa}_1[\mathbf{x}_1,...,\mathbf{x}_N],...,\mathbf{sa}_N[\mathbf{x}_1,...,\mathbf{x}_N]$, each of which is also of size $D\times 1$. </mark>

$D$ is the embedding dimension of each token and $N$ is the number of input tokens. 

Then the nth output $\mathbf{sa}_{n}[\mathbf{x}_{1},\ldots, \mathbf{x}_{N}]$ is a weighted sum of all the values $\mathbf{v}_{1},\ldots,\mathbf{v}_{N}$:

\begin{eqnarray}\label{eq:transformer_sattention1}
  \mbox{{\bf sa}}_{n}[\mathbf{x}_{1},\ldots, \mathbf{x}_{N}] = \sum_{m=1}^{N}a[\mathbf{x}_{m}, \mathbf{x}_{n}]\mathbf{v}_{m}.
 \end{eqnarray}

The scalar weight a[$\mathbf{x}_{m},\mathbf{x}_{n}$] is the attention that the nth output pays to input $\mathbf{x}_{m}$. 

The $N$ weights $a[•,\mathbf{x}_{n}]$ are non-negative and sum to one. 

Hence, self-attention can be thought of as routing the values in different proportions to create each output (figure 12.1).

The attention weights $a[\mathbf{x}_m,\mathbf{x}_n]$ combine the values from different inputs. 

They are also sparse since there is only one weight for each ordered pair of inputs $(\mathbf{x}_m,\mathbf{x}_n)$, regardless of the size of these inputs (figure 12.2c). 

It follows that the number of attention weights has a quadratic dependence on the sequence length N, but is independent of the length D of each input.

---

<img src="assets/Chap12/TransformerRouting.svg" style="filter: invert(1);" width="100%">

Self-attention as routing. The self-attention mechanism takes N
inputs $\mathbf{x}_1, . . . , \mathbf{x}_N ∈ \mathbb{R}^{D}$ (here N = 3 and D = 4) and processes each separately to compute N value vectors. 

The nth output $\mathbf{sa}_{n}[\mathbf{x}_1, . . . , \mathbf{x}_N]$ (written as $\mathbf{sa}_{n}[\mathbf{x}_•]$
for short) is then computed as a weighted sum of the N value vectors, where the
weights are positive and sum to one. 

a) Output $\mathbf{sa}_{1}[\mathbf{x}_•]$ is computed as $a[\mathbf{x}_1, \mathbf{x}_1] =
0.1$ times the first value vector, $a[\mathbf{x}_2, \mathbf{x}_1] = 0.3$ times the second value vector, and $a[\mathbf{x}_3, \mathbf{x}_1] = 0.6$ times the third value vector.

b) Output $\mathbf{sa}_{2}[\mathbf{x}_•]$ is computed in the same way, but this time with weights of $0.5, 0.2$, and $0.3$. 

c) The weighting for output $\mathbf{sa}_{3}[\mathbf{x}_•]$ is different again. Each output can hence be thought of as a different routing of the N values. 

---



<img src="assets/Chap12/TransformerSA1.svg" style="filter: invert(1);" width="100%">

Self-attention for N =3 inputs $\mathbf{x}_n$, each with dimension D=4. 

a) Each input $\mathbf{x}_m$ is operated on independently by the same weights $\Omega_v$ (same color equals same weight) and biases $\beta_v$ (not shown) to form the values $\beta_v + \Omega_v\mathbf{x}_m$. 

    Each output is a linear combination of the values, with the attention weight $a[\mathbf{x}_m, \mathbf{x}_n]$ defining the contribution of the $m$th value to the $n$th output. 

b) Matrix showing block sparsity of linear transformation $\Omega_v$ between inputs and values. 

c) Matrix showing sparsity of attention weights relating values and outputs.

---



### Query vectors, Key vectors and Attention Weights

In the previous section, we saw that the outputs result from two chained linear transformations; the value vectors $\beta_v + \Omega_v \mathbf{x}_m$ are computed independently for each input $\mathbf{x}_m$, and these vectors are combined linearly by the attention weights $a[\mathbf{x}_m,\mathbf{x}_n]$. 

<!-- However, the overall self-attention computation is nonlinear. 

As we’ll see shortly, the attention weights are themselves nonlinear functions of the input. 

This is an example of a hypernetwork, where one network branch computes the weights of another.  -->

To compute the attention, we apply two more linear transformations to the inputs:


\begin{eqnarray}
  \mathbf{q}_{n} &=& \boldsymbol\beta_{q}+\boldsymbol\Omega_{q}\mathbf{x}_{n}\nonumber \\
  \mathbf{k}_{m} &=& \boldsymbol\beta_{k}+\boldsymbol\Omega_{k}\mathbf{x}_{m},
 \end{eqnarray}

where $\mathbf{q}_n$ and $\mathbf{k}_m$ are termed queries and keys, respectively. Then we compute dot products between the queries and keys and pass the results through a softmax function:


\begin{eqnarray}\label{eq:transformer_sattention2}
  a[\mathbf{x}_{m},\mathbf{x}_{n}] &=& \mbox{softmax}_{m}\left[\mathbf{k}_{\bullet}^{T}\mathbf{q}_{n}\right]\nonumber\\
  \\
  &=& \frac{\exp\left[\mathbf{k}_{m}^{T}\mathbf{q}_{n}\right]}{\sum_{m'=1}^{N}\exp\left[\mathbf{k}_{m'}^{T}\mathbf{q}_{n} \right]},
 \end{eqnarray}

so for each $\mathbf{x}_n$, they are positive and sum to one (figure 12.3). For obvious reasons, this is known as <u>dot-product self-attention</u>.

The names “queries” and “keys” were inherited from the field of information retrieval
and have the following interpretation: the dot product operation returns a measure of
similarity between its inputs, so the weights $a[\mathbf{x}_\bullet,\mathbf{x}_n]$ depend on the relative similarities between the nth query and all of the keys. 

The softmax function means that the key vectors “compete” with one another to contribute to the final result. 

The queries and keys must have the same dimensions. 

However, these can differ from the dimension of the values,  which is usually the same size as the input, so the representation doesn’t change size.

<img src="assets/Chap12/TransformerSA2.svg" style="filter: invert(1);" width="100%">

Computing attention weights. a) Query vectors qn = βq + Ωqxn
and key vectors kn = βk + Ωkxn are computed for each input xn. b) The dot
products between each query and the three keys are passed through a softmax
function to form non-negative attentions that sum to one. c) These route the
value vectors (figure 12.1) via the sparse matrix from figure 12.2c.

### Why separate query and key vectors?

The separation of query and key vectors allows the model to learn different representations for the purpose of matching queries to keys. 

If the same vector were used for both queries and keys, the model would be limited to computing similarities based on a single representation. 

By having separate projections, the model can learn to emphasize different aspects of the input when computing attention scores, leading to more flexible and expressive attention mechanisms.

Following examples illustrate how separate query and key vectors enable more flexible attention patterns.

1. Alice told Bob that <mark>he</mark> should leave because <mark>he</mark> was tired.

    Focus on the first _he_.
  
  * Query: What does it seek? The first _he_ seeks information about which male person it refers to, potentially Bob.
  
  * Key: What does it offer? The first _he_ can serve as a useful reference for the second he, which may refer to the same person.

2. The <mark>cat</mark> with the white paws <mark>is</mark> sleeping.

    Focus on cat.

  * Query: The noun cat may attend to white, paws, and the surrounding phrase to develop a richer representation of which cat is being discussed.
  
  * Key: The noun cat offers grammatical information to is, including that the subject is singular. Similarly, cat imposes constraints on what can show up later in the sentence. For example, it rules out wings as a possible continuation, since cats do not have wings.

3. The <mark>chef</mark> carefully <mark>prepared</mark> the <mark>meal</mark>.

    Focus on prepared.
  
  * Query: The verb may seek its subject (chef) to determine who performed the action.

  * Key: The verb can offer information to meal, identifying the action performed on it.


### Non-linearity in self-attention


Self-attention does not require an explicit activation function such as ReLU to introduce non-linearity. Although the queries, keys, and values are computed using linear transformations, the attention weights depend nonlinearly on the inputs.

Non-linearity enters in two places:

- **Dot-product scores.** Each score combines a query and a key, both of which depend on $\mathbf{X}$. Consequently, $\mathbf{Q}\mathbf{K}^{\mathsf T}$ is quadratic in the input, rather than linear.
- **Softmax weights.** Softmax exponentiates the scores and normalizes them so that the weights in each row sum to one. This is a nonlinear transformation of the scores.

Each output is a weighted sum of value vectors:

$$
\mathbf{y}_i=\sum_j \alpha_{ij}(\mathbf{X})\,\mathbf{v}_j.
$$

For fixed weights $\alpha_{ij}$, this sum would be linear in the values. In self-attention, however, the weights themselves change with the input. Changing a word can therefore change both its value vector and how information is combined across the sequence. This dependence makes the overall mapping nonlinear.


### Matrix form

The above computation can be written in a compact form if the N inputs $\mathbf{x}_n$ form the columns of the D×N matrix $\mathbf{X}$. The values, queries, and keys can be computed as:

\begin{eqnarray}
 \mathbf{V}[\mathbf{X}] &=& \boldsymbol\beta_{v}\mathbf{1}^{T}+\boldsymbol\Omega_{\textit{v}}\mathbf{X}\nonumber \\
 \mathbf{Q}[\mathbf{X}] &=& \boldsymbol\beta_{q}\mathbf{1}^{T}+\boldsymbol\Omega_{\textit{q}}\mathbf{X}\nonumber \\
 \mathbf{K}[\mathbf{X}] &=& \boldsymbol\beta_{k}\mathbf{1}^{T}+\boldsymbol\Omega_{\textit{k}}\mathbf{X},
 \end{eqnarray}

where $\mathbf{1}$ is an N ×1 vector containing ones. The self-attention computation is then:

\begin{eqnarray}
  \mbox{{\bf Sa}}[\mathbf{X}] =\mathbf{V}[\mathbf{X}]\cdot\mbox{\bf Softmax}\Bigl[\mathbf{K}[\mathbf{X}]^{T}\mathbf{Q}[\mathbf{X}]\Bigr],
 \end{eqnarray}

where the function $\mbox{\bf Softmax}[•]$ takes a matrix and performs the softmax operation
independently on each of its columns (figure 12.4). In this formulation, we have explicitly included the dependence of the values, queries, and keys on the input X to emphasize
that self-attention computes a kind of triple product based on the inputs. However, from
now on, we will drop this dependence and just write:

\begin{eqnarray}\label{eq:transformer_sa_matrix}
  \mbox{{\bf Sa}}[\mathbf{X}] =\mathbf{V}\cdot\mbox{\bf Softmax}\Bigl[\mathbf{K}^{T}\mathbf{Q}\Bigr].
 \end{eqnarray}


<img src="assets/Chap12/TransformerBlockSA.svg" style="filter: invert(1);" width="100%">

Self-attention in matrix form. Self-attention can be implemented
efficiently if we store the N input vectors xn in the columns of the D×N matrix X.
The input X is operated on separately by the query matrix Q, key matrix K, and
value matrix V. The dot products are then computed using matrix multiplication,
and a softmax operation is applied independently to each column of the resulting
matrix to calculate the attentions. Finally, the values are post-multiplied by the
attentions to create an output of the same size as the input.



In [25]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x_n} \in \mathbb{R}^D $and returns $x^{\prime} \in \mathbb{R}^D$ outputs 
.

In [26]:
# Set seed so we get the same random numbers
np.random.seed(3)
# Number of inputs
N = 3
# Number of dimensions of each input
D = 4
# Create an empty list 
all_x = []
# Create elements x_n and append to list
for n in range(N):
  all_x.append(np.random.normal(size=(D,1)))
# Print out the list
print(all_x)

[array([[ 1.789],
       [ 0.437],
       [ 0.096],
       [-1.863]]), array([[-0.277],
       [-0.355],
       [-0.083],
       [-0.627]]), array([[-0.044],
       [-0.477],
       [-1.314],
       [ 0.885]])]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [27]:
# Set seed so we get the same random numbers
np.random.seed(0)

# Choose random values for the parameters
omega_q = np.random.normal(size=(D,D))
omega_k = np.random.normal(size=(D,D))
omega_v = np.random.normal(size=(D,D))
beta_q  = np.random.normal(size=(D,1))
beta_k  = np.random.normal(size=(D,1))
beta_v  = np.random.normal(size=(D,1))

Now let's compute the queries, keys, and values for each input

In [28]:
# Make three lists to store queries, keys, and values
all_queries = []
all_keys    = []
all_values  = []
# For every input
for x in all_x:
  # Compute the keys, queries and values.
  query = (omega_q @ x) + beta_q
  key   = (omega_k @ x) + beta_k
  value = (omega_v @ x) + beta_v

  all_queries.append(query)
  all_keys.append(key)
  all_values.append(value)

We'll need a softmax function (equation 12.5) -- here, it will take a list of arbitrary numbers and return a list where the elements are non-negative and sum to one

In [29]:
def softmax(items_in):

  # Compute the elements of items_out
  items_out = np.exp(items_in - np.max(items_in))
  items_out /= np.sum(items_out)

  return items_out ;

Now compute the self attention values:

In [30]:
# Create emptymlist for output
all_x_prime = []

# For each output
for n in range(N):
  # Create list for dot products of query N with all keys
  all_km_qn = []
  # Compute the dot products
  for key in all_keys:
    # Compute the appropriate dot product
    dot_product = all_queries[n].T @ key

    # Store dot product
    all_km_qn.append(dot_product)

  # Compute dot product
  attention = softmax(all_km_qn)
  # Print result (should be positive sum to one)
  print("Attentions for output ", n)
  print(attention)

  # Compute a weighted sum of all of the values according to the attention
  x_prime = np.zeros_like(all_values[0])
  for m in range(N):
    x_prime += attention[m] * all_values[m]

  all_x_prime.append(x_prime)


# Print out true values to check you have it correct
print("x_prime_0_calculated:", all_x_prime[0].transpose())
print("x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]")
print("x_prime_1_calculated:", all_x_prime[1].transpose())
print("x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]")
print("x_prime_2_calculated:", all_x_prime[2].transpose())
print("x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]")

Attentions for output  0
[[[1.243e-13]]

 [[9.983e-01]]

 [[1.719e-03]]]
Attentions for output  1
[[[2.795e-12]]

 [[5.855e-03]]

 [[9.941e-01]]]
Attentions for output  2
[[[0.005]]

 [[0.007]]

 [[0.988]]]
x_prime_0_calculated: [[ 0.947 -0.243 -0.913 -0.445]]
x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]
x_prime_1_calculated: [[ 1.642 -0.085  4.028  2.187]]
x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]
x_prime_2_calculated: [[ 1.619 -0.066  3.969  2.159]]
x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]


Now let's compute the same thing, but using matrix calculations. We'll store the 
 inputs 
 in the columns of a 
 matrix, using equations 12.6 and 12.7/8.

Note: The book uses column vectors (for compatibility with the rest of the text), but in the wider literature it is more normal to store the inputs in the rows of a matrix; in this case, the computation is the same, but all the matrices are transposed and the operations proceed in the reverse order.

In [31]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax
     

 # Now let's compute self attention in matrix form
def self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):

  # 1. Compute queries, keys, and values
  # 2. Compute dot products
  # 3. Apply softmax to calculate attentions
  # 4. Weight values by attentions
  # Replace this line
  all_queries = (omega_q @ X) + beta_q
  all_keys    = (omega_k @ X) + beta_k
  all_values  = (omega_v @ X) + beta_v
  # X_prime = all_values @ softmax_cols(all_keys.T @ all_queries)
  X_prime = all_values @ softmax_cols(all_keys.T @ all_queries / np.sqrt(all_keys.shape[0]))


  return X_prime
     

# Copy data into matrix
X = np.zeros((D, N))
X[:,0] = np.squeeze(all_x[0])
X[:,1] = np.squeeze(all_x[1])
X[:,2] = np.squeeze(all_x[2])

# Run the self attention mechanism
X_prime = self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

# Print out the results
print(X_prime)
     

[[ 0.974  1.596  1.326]
 [-0.237 -0.095  0.131]
 [-0.723  3.702  3.024]
 [-0.344  2.013  1.69 ]]


If you did this correctly, the values should be the same as above.

TODO:

Print out the attention matrix You will see that the values are quite extreme (one is very close to one and the others are very close to zero. Now we'll fix this problem by using scaled dot-product attention.

In [32]:
# Now let's compute self attention in matrix form
def scaled_dot_product_self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):

  # 1. Compute queries, keys, and values
  # 2. Compute dot products
  # 3. Scale the dot products as in equation 12.9
  # 4. Apply softmax to calculate attentions
  # 5. Weight values by attentions
  # Replace this line
  all_queries = (omega_q @ X) + beta_q
  all_keys    = (omega_k @ X) + beta_k
  all_values  = (omega_v @ X) + beta_v

  X_prime = np.zeros_like(X)

  # Compute the dot products
  dot_products = all_queries.T @ all_keys
  # Scale the dot products
  scale_factor = np.sqrt(all_keys.shape[0])
  scaled_dot_products = dot_products / scale_factor
  # Apply softmax to calculate attentions
  attentions = softmax_cols(scaled_dot_products)
  # Weight values by attentions
  X_prime = (attentions @ all_values.T).T

  return X_prime
     

# Run the self attention mechanism
X_prime = scaled_dot_product_self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

# Print out the results
print(X_prime) 

[[ 0.138  1.481 -1.737]
 [-0.035 -0.175  3.436]
 [-0.131  1.996 -5.66 ]
 [-0.064  1.109 -2.328]]
